# Análise Exploratória — Titanic

## Objetivo

Investigar o perfil dos passageiros e os padrões de sobrevivência no Titanic, utilizando as tabelas Gold para comparar grupos, identificar tendências e produzir insights sustentados pelos dados.

## Etapas da análise

1. **Conferência dos resultados:** verificar a consistência dos indicadores e das agregações da Gold.
2. **Indicadores gerais:** analisar o total de passageiros, os resultados de sobrevivência e a idade média.
3. **Análise por sexo:** comparar quantidade de passageiros, idade média e taxa de sobrevivência.
4. **Sobrevivência por faixa etária:** comparar os resultados entre as faixas etárias definidas.
5. **Sobrevivência por classe:** avaliar as diferenças entre as três classes de passageiros.
6. **Sobrevivência por porto de embarque:** comparar os resultados por local de embarque.
7. **Sobrevivência por deck:** analisar os resultados por deck identificado e pela categoria desconhecida.
8. **Sobrevivência por classe e sexo:** investigar diferenças entre os grupos combinados.
9. **Exploração complementar:** avaliar outras variáveis, quando contribuírem para a compreensão dos resultados.
10. **Consolidação dos insights:** registrar as principais descobertas e suas limitações.

## Fontes de dados

- **Camada Gold:** indicadores gerais e tabelas agregadas para as análises comparativas.
- **Camada Silver:** dados tratados no nível do passageiro, utilizados quando uma investigação complementar exigir maior detalhamento.

## Metodologia

- Conferir a consistência das métricas antes de interpretá-las.
- Comparar quantidades e taxas de sobrevivência.
- Considerar o tamanho dos grupos ao interpretar percentuais.
- Investigar padrões sem confundir associação com causalidade.
- Registrar conclusões sustentadas pelos resultados observados.

## Limitações

A análise utiliza o dataset Titanic, com 891 registros. As idades ausentes foram preenchidas na Silver pela mediana de cada combinação de classe e sexo. A categoria de deck desconhecido representa registros originalmente sem informação de cabine. Essas decisões devem ser consideradas na interpretação dos resultados.

## Resultado esperado

Produzir uma análise exploratória reproduzível, com comparações e insights documentados. Os resultados servirão de base para a construção do relatório interativo no Power BI e para a documentação do projeto no GitHub.

In [0]:
# 1. Indicadores gerais dos passageiros

df_eda_indicadores_gerais = spark.table(
    "projeto_titanic.gold.indicadores_gerais"
)

display(df_eda_indicadores_gerais)

total_passageiros,total_sobreviventes,total_nao_sobreviventes,idade_media_geral,percentual_sobreviventes,percentual_nao_sobreviventes
891,342,549,29.088855218855215,38.38,61.62


In [0]:
# 2. Conferência de consistência das tabelas Gold
from pyspark.sql.functions import sum as spark_sum

indicadores = df_eda_indicadores_gerais.first()

referencia = (
    indicadores["total_passageiros"],
    indicadores["total_sobreviventes"],
    indicadores["total_nao_sobreviventes"]
)

tabelas_gold = [
    ("Sexo", "projeto_titanic.gold.analise_por_sexo"),
    ("Faixa etária", "projeto_titanic.gold.sobrevivencia_por_faixa_etaria"),
    ("Classe", "projeto_titanic.gold.sobrevivencia_por_classe"),
    ("Porto de embarque", "projeto_titanic.gold.sobrevivencia_por_porto_embarque"),
    ("Deck", "projeto_titanic.gold.sobrevivencia_por_deck"),
    ("Classe e sexo", "projeto_titanic.gold.sobrevivencia_por_classe_sexo")
]

validacoes = []

for analise, tabela in tabelas_gold:
    totais = (
        spark.table(tabela)
        .agg(
            spark_sum("total_passageiros").alias("passageiros"),
            spark_sum("total_sobreviventes").alias("sobreviventes"),
            spark_sum("total_nao_sobreviventes").alias("nao_sobreviventes")
        )
        .first()
    )

    valores = (
        totais["passageiros"],
        totais["sobreviventes"],
        totais["nao_sobreviventes"]
    )

    status = "OK" if valores == referencia else "Verificar"

    validacoes.append((analise, *valores, status))

df_validacao_gold = spark.createDataFrame(
    validacoes,
    [
        "analise",
        "total_passageiros",
        "total_sobreviventes",
        "total_nao_sobreviventes",
        "status"
    ]
)

display(df_validacao_gold)

analise,total_passageiros,total_sobreviventes,total_nao_sobreviventes,status
Sexo,891,342,549,OK
Faixa etária,891,342,549,OK
Classe,891,342,549,OK
Porto de embarque,891,342,549,OK
Deck,891,342,549,OK
Classe e sexo,891,342,549,OK


In [0]:
# 3. Análise exploratória por sexo

df_eda_analise_por_sexo = spark.table(
    "projeto_titanic.gold.analise_por_sexo"
)

display(df_eda_analise_por_sexo.orderBy("sexo"))

sexo,total_passageiros,total_sobreviventes,total_nao_sobreviventes,idade_media,taxa_sobrevivencia
feminino,314,233,81,27.194267515923567,74.2
masculino,577,109,468,30.119878682842284,18.89


### Insight — Sobrevivência por sexo

A taxa de sobrevivência foi de **74,20% entre as mulheres** e **18,89% entre os homens**, uma diferença de 55,31 pontos percentuais.

Embora as mulheres representassem 314 dos 891 passageiros, houve 233 sobreviventes nesse grupo. Entre os homens, foram 109 sobreviventes de um total de 577 passageiros.

Os resultados indicam uma associação expressiva entre sexo e sobrevivência nessa base. A análise não demonstra causalidade; a classe dos passageiros, entre outras características, também pode estar relacionada às diferenças observadas.

In [0]:
# 4. Análise exploratória por faixa etária

df_eda_sobrevivencia_faixa_etaria = (
    spark.table("projeto_titanic.gold.sobrevivencia_por_faixa_etaria")
)

display(df_eda_sobrevivencia_faixa_etaria)

faixa_etaria,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
00-12 | Crianças,69,40,29,57.97
13-17 | Adolescentes,44,21,23,47.73
18-29 | Adultos jovens,409,131,278,32.03
30-59 | Adultos,343,143,200,41.69
60+ | Idosos,26,7,19,26.92


### Insight — Sobrevivência por faixa etária

As crianças apresentaram a maior taxa de sobrevivência (57,97%), seguidas pelos adolescentes (47,73%). Entre adultos jovens, a taxa foi de 32,03%, enquanto adultos de 30 a 59 anos apresentaram 41,69%. Os idosos tiveram a menor taxa (26,92%), mas esse grupo reúne apenas 26 passageiros.

Os resultados sugerem diferenças de sobrevivência entre as faixas etárias, sem uma relação linear entre idade e sobrevivência. A interpretação exige cautela, especialmente nos grupos menores. Além disso, as idades ausentes foram preenchidas na Silver por medianas calculadas segundo classe e sexo, o que pode influenciar a classificação de alguns passageiros nas faixas etárias.

In [0]:
# 5. Análise exploratória por classe

df_eda_sobrevivencia_classe = spark.table(
    "projeto_titanic.gold.sobrevivencia_por_classe"
)

display(df_eda_sobrevivencia_classe)

classe,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
1,216,136,80,62.96
2,184,87,97,47.28
3,491,119,372,24.24


### Insight — Sobrevivência por classe

A primeira classe apresentou a maior taxa de sobrevivência (62,96%), seguida pela segunda classe (47,28%) e pela terceira classe (24,24%).

A terceira classe concentrava 491 passageiros, aproximadamente 55% do total, e registrou 372 não sobreviventes. A diferença de 38,72 pontos percentuais entre a primeira e a terceira classe evidencia uma associação expressiva entre classe da passagem e sobrevivência nesta base.

Os resultados não demonstram causalidade isoladamente. Fatores como sexo e idade também podem estar relacionados às diferenças observadas entre as classes.

In [0]:
# 6. Análise exploratória por porto de embarque

df_eda_sobrevivencia_porto = spark.table(
    "projeto_titanic.gold.sobrevivencia_por_porto_embarque"
)

display(df_eda_sobrevivencia_porto)

porto_embarque,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
Cherbourg,168,93,75,55.36
Queenstown,77,30,47,38.96
Southampton,646,219,427,33.9


### Insight — Sobrevivência por porto de embarque

Cherbourg apresentou a maior taxa de sobrevivência (55,36%), seguido por Queenstown (38,96%) e Southampton (33,90%).

Cherbourg registrou 93 sobreviventes entre 168 passageiros, enquanto Southampton teve 219 sobreviventes entre 646 passageiros. A diferença entre as taxas foi de **21,46 pontos percentuais**.

Os resultados indicam uma associação entre o porto de embarque e a sobrevivência, mas não demonstram causalidade. Diferenças no perfil dos passageiros, como classe e sexo, podem ajudar a explicar os resultados observados.

In [0]:
# 7. Análise exploratória por deck

df_eda_sobrevivencia_deck = spark.table(
    "projeto_titanic.gold.sobrevivencia_por_deck"
)

display(df_eda_sobrevivencia_deck.orderBy("deck"))

deck,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
A,15,7,8,46.67
B,47,35,12,74.47
C,59,35,24,59.32
D,33,25,8,75.76
E,32,24,8,75.0
F,13,8,5,61.54
G,4,2,2,50.0
T,1,0,1,0.0
U,687,206,481,29.99


### Insight — Sobrevivência por deck

Entre os decks identificados, o deck D apresentou a maior taxa de sobrevivência (75,76%), seguido por E (75,00%) e B (74,47%). Esses grupos, porém, possuem poucos passageiros, o que exige cautela na comparação.

A categoria `U` concentra 687 passageiros (77,1% da base) e apresentou taxa de sobrevivência de 29,99%. Como a maior parte dos registros não possui informação de cabine, os resultados por deck têm representatividade limitada e não permitem conclusões definitivas sobre a relação entre deck e sobrevivência.

In [0]:
# 8. Análise exploratória por classe e sexo

df_eda_sobrevivencia_classe_sexo = spark.table(
    "projeto_titanic.gold.sobrevivencia_por_classe_sexo"
)

display(df_eda_sobrevivencia_classe_sexo)

classe,sexo,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
1,feminino,94,91,3,96.81
1,masculino,122,45,77,36.89
2,feminino,76,70,6,92.11
2,masculino,108,17,91,15.74
3,feminino,144,72,72,50.0
3,masculino,347,47,300,13.54


### Insight — Sobrevivência por classe e sexo

- **1ª classe:** 96,81% das mulheres sobreviveram, contra 36,89% dos homens.
- **2ª classe:** 92,11% das mulheres sobreviveram, contra 15,74% dos homens.
- **3ª classe:** 50,00% das mulheres sobreviveram, contra 13,54% dos homens.

As mulheres apresentaram taxas de sobrevivência superiores às dos homens em todas as classes, a maior diferença entre os sexos ocorreu na segunda classe, com 76,37 pontos percentuais. Os resultados indicam diferenças de sobrevivência associadas ao sexo e à classe da passagem, mas não demonstram causalidade.

In [0]:
# 9. Análise complementar por situação familiar

from pyspark.sql.functions import col, when, count, sum, round

df_eda_titanic_silver = spark.table(
    "projeto_titanic.silver.titanic"
)

df_eda_sobrevivencia_familia = (
    df_eda_titanic_silver
    .withColumn(
        "situacao_familiar",
        when(
            (col("irmaos_conjuges") == 0) & (col("pais_filhos") == 0),
            "Sem familiares registrados"
        ).otherwise("Com familiares registrados")
    )
    .groupBy("situacao_familiar")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
)

display(df_eda_sobrevivencia_familia)

situacao_familiar,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
Com familiares registrados,354,179,175,50.56
Sem familiares registrados,537,163,374,30.35


In [0]:
# 9. Análise complementar por tamanho da família

from pyspark.sql.functions import col, when, count, sum, round

df_eda_sobrevivencia_familia = (
    spark.table("projeto_titanic.silver.titanic")
    .withColumn(
        "tamanho_familia",
        col("irmaos_conjuges") + col("pais_filhos") + 1
    )
    .withColumn(
        "grupo_tamanho_familia",
        when(col("tamanho_familia") == 1, "1 pessoa - Sozinho")
        .when(col("tamanho_familia") <= 4, "2 a 4 pessoas - Família pequena")
        .otherwise("5+ pessoas - Família grande")
    )
    .groupBy("grupo_tamanho_familia")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("grupo_tamanho_familia")
)

display(df_eda_sobrevivencia_familia)

grupo_tamanho_familia,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
1 pessoa - Sozinho,537,163,374,30.35
2 a 4 pessoas - Família pequena,292,169,123,57.88
5+ pessoas - Família grande,62,10,52,16.13


### Insight — Sobrevivência por tamanho da família

- **Sozinhos:** taxa de sobrevivência de 30,35%.
- **Famílias pequenas (2 a 4 pessoas):** taxa de sobrevivência de 57,88%.
- **Famílias grandes (5 ou mais pessoas):** taxa de sobrevivência de 16,13%.

Passageiros em famílias pequenas apresentaram a maior taxa de sobrevivência, enquanto aqueles em famílias grandes tiveram a menor. A diferença entre esses grupos foi de 41,75 pontos percentuais.

Os resultados indicam uma associação entre tamanho da família e sobrevivência, mas não demonstram causalidade. O grupo familiar considera apenas o passageiro e os irmãos, cônjuges, pais e filhos registrados na base. Diferenças de classe e sexo também podem influenciar os resultados.

## Síntese dos principais insights

- **Visão geral:** dos 891 passageiros analisados, 342 sobreviveram, correspondendo a 38,38% da base.
- **Sexo:** a taxa de sobrevivência foi de 74,20% entre as mulheres e 18,89% entre os homens, uma diferença de 55,31 pontos percentuais.
- **Classe:** a primeira classe apresentou taxa de sobrevivência de 62,96%, enquanto a terceira apresentou 24,24%.
- **Idade:** crianças tiveram a maior taxa de sobrevivência entre as faixas etárias analisadas (57,97%). A relação entre idade e sobrevivência não foi linear.
- **Classe e sexo:** as mulheres apresentaram taxas de sobrevivência superiores às dos homens nas três classes. A maior diferença ocorreu na segunda classe, com 76,37 pontos percentuais.
- **Tamanho da família:** passageiros em grupos familiares de 2 a 4 pessoas apresentaram taxa de sobrevivência de 57,88%, contra 30,35% entre os que viajavam sozinhos e 16,13% nos grupos de 5 ou mais pessoas.
- **Porto de embarque:** Cherbourg apresentou a maior taxa de sobrevivência (55,36%), seguido por Queenstown (38,96%) e Southampton (33,90%).

## Limitações da análise

Os resultados descrevem associações observadas no dataset e não demonstram causalidade. As diferenças entre os grupos podem estar relacionadas a fatores combinados, como sexo, idade e classe da passagem.

As idades ausentes foram preenchidas na Silver por medianas calculadas conforme classe e sexo. Além disso, 687 passageiros (77,1% da base) não possuíam informação de cabine, limitando a interpretação das taxas por deck. Os grupos com poucos passageiros também exigem cautela na comparação.